In [1]:
import torch
from PIL import Image
import os
import numpy as np
from transformers import CLIPProcessor, CLIPModel
import matplotlib.pyplot as plt
import yaml
from cleanfid import fid
import lpips
import torchvision.transforms as T
import itertools
import random
import collections
import re

with open(os.path.expanduser("/work/cvcs2026/stochastic_parrots/config.yaml"), "r") as f:
    config = yaml.safe_load(f)

/homes/saresta/cvcs2026/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Definition Output Directories

In [3]:
# Define the version of LoRA being evaluated (v1 or v2)
LORA_VERSION = 2

MODEL_PATH = config["paths"]["base_model_dir"]
LORA_PATH = config["paths"]["lorav" + str(LORA_VERSION) + "_model_dir"]
LORA_WEIGHTS_FILE = config["weights_file"]["lora_weights"]

REF_IMAGES_DIR = config["paths"]["instance_images_dir"]

OUTPUT_CLIPI_SDXL = config["paths"]["evaluation_dir"] + "/metrics/sdxl/clipi"
OUTPUT_CLIPI_LORA = config["paths"]["evaluation_dir"] + "/metrics/lora-v" + str(LORA_VERSION) + "/clipi"

OUTPUT_CLIPT_SDXL = config["paths"]["evaluation_dir"] + "/metrics/sdxl/clipt"
OUTPUT_CLIPT_LORA = config["paths"]["evaluation_dir"] + "/metrics/lora-v" + str(LORA_VERSION) + "/clipt"

OUTPUT_FID_SDXL = config["paths"]["evaluation_dir"] + "/metrics/sdxl/fid"
OUTPUT_FID_LORA = config["paths"]["evaluation_dir"] + "/metrics/lora-v" + str(LORA_VERSION) + "/fid"

OUTPUT_LPIPS_SDXL = config["paths"]["evaluation_dir"] + "/metrics/sdxl/lpips"
OUTPUT_LPIPS_LORA = config["paths"]["evaluation_dir"] + "/metrics/lora-v" + str(LORA_VERSION) + "/lpips"

PROMPTS_DIR = config["paths"]["prompts_dir"]


model_id = "openai/clip-vit-large-patch14"  # Model version: ViT Large (24 layers) with patch14 (each patch is 14x14 pixels)
model = CLIPModel.from_pretrained(model_id).to("cuda")  # Load the CLIP model (pretrained) and move it to the specified device (GPU or CPU)
processor = CLIPProcessor.from_pretrained(model_id)  # Load the CLIP processor to automatically preprocess images and text for the model

Loading weights: 100%|██████████| 590/590 [00:00<00:00, 1869.00it/s]
CLIPModel LOAD REPORT from: openai/clip-vit-large-patch14
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


# CLIP-I Metric
Test for "Personalization Quality". How well the model captures the target concept? 

We use this metric to measure the distance between, the generated images and the reference images.

## Helper Functions

In [ ]:
def get_features(image_paths):
    images = [Image.open(p).convert("RGB") for p in image_paths]  # Load and convert images to RGB (to be sure)
    inputs = processor(images=images, return_tensors="pt").to("cuda") # Transform images into tensors (pytorch) and move to DEVICE
    
    with torch.no_grad():
        outputs = model.get_image_features(**inputs) # Get the image features from the CLIP model (vector of 1024 numbers)
        
        # 1. Extract the tensor from the model output (it can be a tensor diretly or a BaseModelOutputWithPooling, depends on the version of model)
        if torch.is_tensor(outputs):
            features = outputs
        else:
            # If it's a BaseModelOutputWithPooling, we look for the right field
            # In CLIP it's usually called 'image_embeds' or it's the first element
            features = getattr(outputs, "image_embeds", outputs[0])

        # 2. If the model output is 3D (Batch, Patches, Features), it outputs the features of all the patches. We need only "CLS Token" (the first patch)
        if len(features.shape) == 3:
            features = features[:, 0, :]

        # 3. Normalize the features to have unit norm (this is important for cosine similarity)
        features = features / features.norm(p=2, dim=-1, keepdim=True)
        
    return features


# Create a "footprint" of our reference images by averaging their features. This will be our "reference feature" to compare against generated images.

ref_paths = [os.path.join(REF_IMAGES_DIR, f) for f in os.listdir(REF_IMAGES_DIR) if f.endswith(('.png', '.jpg'))] # Get all image paths in the reference directory
ref_features = get_features(ref_paths)  # Get the features of all reference images (matrix of size [num_ref_images, feature_dim])
avg_ref_feature = ref_features.mean(dim=0, keepdim=True)  # Average the features to get a single "reference feature" (vector of size [1, feature_dim])
avg_ref_feature /= avg_ref_feature.norm(p=2, dim=-1, keepdim=True) # Normalize the average reference feature to have unit norm (important for cosine similarity)


# This function evaluates the fidelity of generated images by computing the cosine similarity. It returns the mean and std of the similarities for all generated images
def evaluate_folder(folder_path):
    gen_paths = [os.path.join(folder_path, f) for f in os.listdir(folder_path) if f.endswith('.png')]  # Get all generated image paths in the specified folder
    if not gen_paths: return 0
    
    all_similarities = []
    for path in gen_paths:
        gen_feat = get_features([path])  # Get the feature of the generated image (vector of size [1, feature_dim])
        sim = (gen_feat @ avg_ref_feature.T).item()  # Compute the cosine similarity between the generated image feature and the average reference feature (scalar)
        all_similarities.append(sim) # Append the similarity to the list of all similarities
    
    return np.mean(all_similarities), np.std(all_similarities)  # Return mean and std of the similarities for all generated images in the folder

## Execution

In [5]:
#print("Compute similarity for Base folder...")
mean_lora, std_lora = evaluate_folder(OUTPUT_CLIPI_LORA)

#print("Compute similarity for LoRA folder...")
mean_base, std_base = evaluate_folder(OUTPUT_CLIPI_SDXL)

print("\n--- RESULTS SUBJECT FIDELITY ---")
print(f"{'Model':<25} | {'Mean':<15} | {'Std':<15}")

print(f"{'Base':<25} | {mean_base:<15.4f} | {std_base:<15.4f}")
print(f"{'LoRA':<25} | {mean_lora:<15.4f} | {std_lora:<15.4f}")

print(f"Fidelity increment: {((mean_lora - mean_base) / mean_base) * 100:.2f}%")


--- RESULTS SUBJECT FIDELITY ---
Model                     | Mean            | Std            
Base                      | 0.6215          | 0.1179         
LoRA                      | 0.7117          | 0.1318         
Fidelity increment: 14.51%


# CLIP-T Metric
Test for "Language Drift". How well the model understands the "non-cat" prompts.

We use this metric to measure the distance between the generated images and their prompt

## Helper Functions

In [6]:
def load_prompts(metric, model):
    base_prompt_dir = os.path.join(PROMPTS_DIR, metric)
    model_dir = os.path.join(base_prompt_dir, model)

    if os.path.isdir(model_dir):
        prompt_file = os.path.join(model_dir, "prompt.txt")
    
    else:
        prompt_file = os.path.join(base_prompt_dir, "prompt.txt")

    print(f"Loading prompts from: {prompt_file}")

    with open(prompt_file, "r") as f:
        prompts = [line.strip() for line in f.readlines() if line.strip()]

    return prompts

prompts = load_prompts("clipt", "sdxl")

Loading prompts from: /work/cvcs2026/stochastic_parrots/evaluation/prompts/clipt/prompt.txt


In [7]:
def compute_clip_t(image_path, text_prompt):
    img = Image.open(image_path).convert("RGB")
    
    # Processor accepts both text and images, and returns tensors ready for the model
    inputs = processor(
        text=[text_prompt], 
        images=[img], 
        return_tensors="pt", 
        padding=True
    ).to("cuda")
    
    with torch.no_grad():
        outputs = model(**inputs)
        
        # We extract the image and text embeddings from the model's output
        image_embeds = outputs.image_embeds
        text_embeds = outputs.text_embeds
        
        image_embeds /= image_embeds.norm(p=2, dim=-1, keepdim=True)
        text_embeds /= text_embeds.norm(p=2, dim=-1, keepdim=True)
        
        # Cosine similarity is computed as the dot product of the normalized embeddings
        similarity = (text_embeds @ image_embeds.T).item()
        
    return similarity


def evaluate_clipt(folder_path, nprompt):
    files = []
    p = prompts[nprompt]
    n = nprompt+1
    if n < 10: n = f"0{n}"
    for f in os.listdir(folder_path):
        if f.startswith(f"image_{n}"):
            files.append(os.path.join(folder_path, f))

    scores = []
    for file in files:
        score = compute_clip_t(file, p)
        scores.append(score)

    return np.mean(scores), np.std(scores)

## Execution

In [8]:
results = []
print(f"{'Prompt':<25} | {'Base':<15} | {'LoRA':<15} | {'Drift (%)':<10}")
print("-" * 75)
for i, prompt in enumerate(prompts):
    mean_base, std_base = evaluate_clipt(OUTPUT_CLIPT_SDXL, i)
    mean_lora, std_lora = evaluate_clipt(OUTPUT_CLIPT_LORA, i)

    current_drift = (1 - (mean_lora / mean_base)) * 100 if mean_base != 0 else 0

    results.append({
        "prompt": prompt,
        "base": mean_base,
        "lora": mean_lora,
        "drift": current_drift
    })

    print(f"{prompt[:25]:<25} | {mean_base:.4f} ± {std_base:.4f} | {mean_lora:.4f} ± {std_lora:.4f} | {current_drift:.2f}%")

avg_base = np.mean([r["base"] for r in results])
avg_lora = np.mean([r["lora"] for r in results])
avg_drift = (1 - (avg_lora / avg_base)) * 100

print("-"*75)
print(f"Average CLIP-T Base: {avg_base:.4f}, Average CLIP-T LoRA: {avg_lora:.4f}, Average Drift: {avg_drift:.2f}%")

Prompt                    | Base            | LoRA            | Drift (%) 
---------------------------------------------------------------------------


A photo of a black cat    | 0.2754 ± 0.0217 | 0.2957 ± 0.0117 | -7.38%
A photo of a dog          | 0.2345 ± 0.0114 | 0.2471 ± 0.0111 | -5.38%
A photo of a rabbit       | 0.2609 ± 0.0087 | 0.2691 ± 0.0087 | -3.13%
A photo of a fox          | 0.2703 ± 0.0081 | 0.2713 ± 0.0087 | -0.38%
A photo of a lion         | 0.2540 ± 0.0068 | 0.2596 ± 0.0074 | -2.19%
A photo of a tiger        | 0.2513 ± 0.0074 | 0.2613 ± 0.0080 | -3.99%
A mountain landscape      | 0.2460 ± 0.0093 | 0.2505 ± 0.0089 | -1.83%
A portrait of a person    | 0.2197 ± 0.0087 | 0.2281 ± 0.0091 | -3.82%
A city skyline            | 0.2320 ± 0.0167 | 0.2282 ± 0.0140 | 1.60%
A bowl of fruit           | 0.2464 ± 0.0134 | 0.2477 ± 0.0151 | -0.52%
A car on a road           | 0.2263 ± 0.0120 | 0.2326 ± 0.0136 | -2.77%
A cake on a table         | 0.2245 ± 0.0149 | 0.2299 ± 0.0150 | -2.42%
A beach at sunset         | 0.2433 ± 0.0125 | 0.2406 ± 0.0100 | 1.11%
A forest in autumn        | 0.2411 ± 0.0082 | 0.2381 ± 0.0123 | 1.26%
A space s

# FID Metric
Test for Quality. How well the specialized model can generate the image of a cat?. 

We use this metric to measure how different is the distribution of specialized model respect to the distribution of base model

NOTE: The more value is close to 0, the more similar the two sets of images are

## Execution

In [9]:
# Compute FID
score_fid = fid.compute_fid(OUTPUT_FID_SDXL, OUTPUT_FID_LORA, device=torch.device("cuda"))

print(f"FID Score: {score_fid:.4f}")

compute FID between two folders


/homes/saresta/cvcs2026/venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 12 worker processes in total. Our suggested max number of worker in current system is 6, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


Found 5000 images in the folder /work/cvcs2026/stochastic_parrots/evaluation/metrics/sdxl/fid


FID fid : 100%|██████████| 157/157 [01:33<00:00,  1.69it/s]


Found 5000 images in the folder /work/cvcs2026/stochastic_parrots/evaluation/metrics/lora-v2/fid


FID fid : 100%|██████████| 157/157 [01:31<00:00,  1.71it/s]


FID Score: 65.5353


# LPIPS Metric
Test for "Mode Collapse". How different are the images generated by a model?

We use this metric to compare each image with the others, and compute how different they are

## Helper functions

In [10]:
loss_fn_alex = lpips.LPIPS(net='alex').to("cuda")

def load_and_preprocess(path):
    img = Image.open(path).convert("RGB")
    transform = T.Compose([
        T.Resize((256, 256)),
        T.ToTensor(),
        T.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)) 
    ])
    return transform(img).unsqueeze(0).to("cuda")

def compute_internal_diversity(folder_path, max_pairs_per_prompt=2000):
    files = [os.path.join(folder_path, f) for f in os.listdir(folder_path) if f.endswith('.png')]
    if not files: return 0

    groups = collections.defaultdict(list)

    # Group images by prompt (assuming filename format: image_{prompt_idx}_*.png)
    for f in files:
        match = re.search(r'image_(\d+)_', f)  # Extract the prompt index from the filename using regex (01, 02, 03, etc.)
        if match:
            prompt_idx = match.group(1)
            groups[prompt_idx].append(os.path.join(folder_path, f))
        else:
            groups["unknown"].append(os.path.join(folder_path, f))

    # We will have groups like {"01": [imgage_01_0001.png, image_01_0002.png, ...], "02": [...], ...}

    all_prompt_distances = {}

    # Compute LPIPS distances for each prompt group, sampling pairs if necessary to limit the number of comparisons
    for prompt_idx, group_files in groups.items():
        if len(group_files) < 2: 
            continue
                    
        all_pairs = list(itertools.combinations(group_files, 2))
        
        if len(all_pairs) > max_pairs_per_prompt:
            sampled_pairs = random.sample(all_pairs, max_pairs_per_prompt)
        else:
            sampled_pairs = all_pairs
            
        # 3. Optimization: Pre-load only the unique files needed for the sampled pairs into GPU memory once
        unique_files = set()
        for a, b in sampled_pairs:
            unique_files.add(a)
            unique_files.add(b)
            
        img_cache = {}
        for f in unique_files:
            img_cache[f] = load_and_preprocess(f)
            
        # 4. Compute LPIPS distances for the sampled pairs using the pre-loaded images
        distances = []
        for path_a, path_b in sampled_pairs:
            img_a = img_cache[path_a]
            img_b = img_cache[path_b]
            
            with torch.no_grad():
                dist = loss_fn_alex(img_a, img_b)
                distances.append(dist.item())
                
        avg_dist = sum(distances) / len(distances)
        all_prompt_distances[prompt_idx] = avg_dist
        print(f"  -> Average LPIPS for {prompt_idx}: {avg_dist:.4f}\n")
        
        # Free GPU memory for the next prompt group
        del img_cache
        torch.cuda.empty_cache()

    if not all_prompt_distances: return 0
    
    # 5. Return the final average across all prompt groups
    final_avg = sum(all_prompt_distances.values()) / len(all_prompt_distances)
    return final_avg

Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]


/homes/saresta/cvcs2026/venv/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/homes/saresta/cvcs2026/venv/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading model from: /homes/saresta/cvcs2026/venv/lib/python3.11/site-packages/lpips/weights/v0.1/alex.pth


## Execution

In [11]:
print("--- SDXL Model ---")

div_base = compute_internal_diversity(OUTPUT_LPIPS_SDXL)

print("\n--- LoRA Model ---")
div_lora = compute_internal_diversity(OUTPUT_LPIPS_LORA)

print("-"*30)
print(f"Internal Diversity (LPIPS) Base: {div_base:.4f}")
print(f"Internal Diversity (LPIPS) LoRA: {div_lora:.4f}")

--- SDXL Model ---
  -> Average LPIPS for 01: 0.5838

  -> Average LPIPS for 02: 0.4577


--- LoRA Model ---
  -> Average LPIPS for 01: 0.5996

  -> Average LPIPS for 02: 0.5563

  -> Average LPIPS for 03: 0.4813

------------------------------
Internal Diversity (LPIPS) Base: 0.5207
Internal Diversity (LPIPS) LoRA: 0.5457
